# Three-seed E. coli RNA encoder comparison

The earlier Newton RiNALMo runs provide the reference for RNAErnie and ERNIE-RNA. Each row reports held-out test accuracy for the same head at seeds 0, 1 and 2 and their arithmetic mean. Gated MIL uses the positive entropy penalty chosen on the historical seed-0 validation data. All six rows use the same test examples.

This saved-results readout uses the setup and reporting cells from [Encoder swaps](../../../encoder_swaps.ipynb). The [full-precision scores](ecoli_ernie_three_seed.json) identify each source fit and record its SHA-256. The underlying experiment includes two heads per RNA encoder. These results compare the encoders and heads on this dataset.

In [1]:
# Files, environment, and elapsed time
import json
import os
import sys
import time
from pathlib import Path

# Set Apple GPU fallback before importing PyTorch; preserve an explicit user setting.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# Numerical arrays and result tables
import numpy as np
import pandas as pd

# Tensor operations, model training, and available compute backends
import torch

# Render notebook tables explicitly rather than relying on an implicit kernel name.
from IPython.display import display

In [2]:
# Search the working directory and its parents for Code/shared_code.
CODE_DIR = next(
    (
        candidate.resolve()
        for parent in (Path.cwd(), *Path.cwd().parents)
        for candidate in (parent, parent / "Code")
        if (candidate / "shared_code").is_dir()
    ),
    None,
)
if CODE_DIR is None:
    raise RuntimeError(
        "Run this notebook from the repository root or a directory inside it "
        "so Code/shared_code can be found."
    )
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Benchmark rows and encoder definitions
from shared_code import datasets
from shared_code.encoders import ENCODERS

# Fusion input streams, prediction-head training, and token-cache verification
from shared_code.fusion import blf, train, swap_results, swap_plan, run_identity
from shared_code.fusion import token_embeddings as tokens

In [3]:
# Table display width (characters) and visible columns, not sequence or token limits.
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

# Preserve the existing preference: Apple MPS, then NVIDIA CUDA, then CPU.
DEVICE = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(
    "torch", torch.__version__,
    "| CUDA available:", torch.cuda.is_available(),
    "| Apple MPS available:", torch.backends.mps.is_available(),
)
if DEVICE == "cuda":
    print(f"Selected device: {DEVICE} | GPU: {torch.cuda.get_device_name()}")
elif DEVICE == "mps":
    print(f"Selected device: {DEVICE} | GPU: Apple GPU (MPS)")
else:
    print(f"Selected device: {DEVICE}")

torch 2.10.0+cu126 | CUDA available: True | Apple MPS available: False
Selected device: cuda | GPU: NVIDIA GeForce RTX 2060 SUPER


In [4]:
# Reading is the default; missing results never enable research computation.
EXPERIMENT = "ecoli_ernie"  # "ecoli_ernie" selects the focused RNA extension.
READ_SAVED_RESULTS = True
RESULT_SET = "newton_h100_seeds012"  # Fresh Newton scores; historical records select baselines only.
RETURNED_WORKSPACE = None  # Optional explicit package root; no automatic newest-run lookup.
REFERENCE_WORKSPACE = None  # Optional returned original Newton cohort for extension comparisons.
from shared_code.fusion.result_sets import resolve_swap_results, resolve_ernie_extension

if EXPERIMENT not in {"full", "ecoli_ernie"}:
    raise ValueError("EXPERIMENT must be 'full' or 'ecoli_ernie'.")
if EXPERIMENT == "ecoli_ernie":
    RESULT_PATHS = resolve_ernie_extension(
        CODE_DIR, workspace=RETURNED_WORKSPACE, reference_workspace=REFERENCE_WORKSPACE,
    )
    REFERENCE_SWAP_ROOT = resolve_swap_results(
        CODE_DIR, "newton_h100_seeds012", workspace=REFERENCE_WORKSPACE,
    ).swaps
else:
    RESULT_PATHS = resolve_swap_results(CODE_DIR, RESULT_SET, workspace=RETURNED_WORKSPACE)
    REFERENCE_SWAP_ROOT = RESULT_PATHS.swaps
if not READ_SAVED_RESULTS:
    RESULT_PATHS.require_writable_fits()
print(f"Result set: {RESULT_PATHS.name}; evidence root: {RESULT_PATHS.root}")

BENCHMARKS = (
    "cov_vac_degradation", "fungal_expression", "ecoli_proteins", "mrna_stability",
)
if EXPERIMENT == "ecoli_ernie":
    BENCHMARKS = ("ecoli_proteins",)
CACHE_ROOT = CODE_DIR / "stage3" / "embeddings"
NEW_CACHE_ROOT = CODE_DIR / "stage4/embeddings/ecoli_ernie"
RESULTS_ROOT = RESULT_PATHS.swaps
ORIGINAL_RESULTS = RESULT_PATHS.originals
COMPARISON_DIR = RESULT_PATHS.analysis_dir
STAGE1_ALIGNMENT = CODE_DIR.parent / "docs" / "figures" / "stability" / "results.json"


Result set: newton_ecoli_ernie; evidence root: D:\academic\UCF\2026_fall\Capstone Project CAP6942\Research\genomics-multimodal-fusion\Code\stage4\results\newton_ecoli_ernie


In [5]:
# Prediction-head training settings
TCONV = "equation"  # Retain the Equation 1 six-mer-to-codon upsampling.
CHANNELS = 100      # Learned filters per TextCNN kernel size, not encoder width.
BATCH_SIZE = 32     # Benchmark rows per training minibatch.
MAX_EPOCHS = 500    # Ceiling; validation-based early stopping may end a fit sooner.
SEED = 0           # Individual-seed displays; in a writable workspace, the seed to fit.
SELECTION_SEED = 0  # Keep validation-selected penalties and baseline choices fixed.
RUN_BOOTSTRAP = False  # Separate analysis; leave off when completing fits or repeats.

# Independent CPU analysis of saved predictions; no fitting or embedding extraction.
RUN_SAVED_COMPARISONS = False
SAVED_REPORT_FILES = {}  # Optional benchmark -> exact report path when several reports match.
COMPARISON_SEEDS = (0, 1, 2)  # One report covers all three; independent of the display SEED.
COMPARISON_DRAWS = 2000
COMPARISON_BOOTSTRAP_SEED = 0
if RUN_SAVED_COMPARISONS and not READ_SAVED_RESULTS:
    raise ValueError(
        "Set READ_SAVED_RESULTS = True for saved-prediction reporting; "
        "training and extraction must remain disabled."
    )

# Execution hooks use the same scientific cells in a separately prepared workspace.
# The launcher requires complete caches and supplies cohort-aware producer/recovery checks.
RUN_TRAINING = False  # The extraction-only launcher sets False; read mode always skips fits.
CACHE_REUSE_ONLY = EXPERIMENT == "ecoli_ernie"
CHECKPOINT_VALIDATOR = None  # None keeps the shared trainer's standard check.
FIT_PRODUCER = None

# Cache chunking and stored-array precision, not encoder or training precision
CHUNK_SIZE = tokens.DEFAULT_CHUNK
DTYPE = np.float16

# Each pair replaces one slot while retaining the other two original encoders.
SWAPS = [
    ("dna", "nt500m"),
    ("dna", "hyenadna"),
    ("rna", "rinalmo"),
    ("rna", "mrnafm"),
    ("rna", "calm"),
    ("prot", "esm2_35m"),
    ("prot", "esm2_150m"),
    ("prot", "protbert"),
]
if EXPERIMENT == "ecoli_ernie":
    SWAPS = [("rna", "rnaernie"), ("rna", "ernierna")]
    if RUN_BOOTSTRAP or RUN_SAVED_COMPARISONS:
        raise ValueError("The focused panel reads head scores only; single-control reports are not configured.")
# Full mode retains all eight replacements; extension mode schedules only its two.
STABILITY_SWAPS = list(SWAPS)
ORIGINAL = dict(blf.ENCODER_OF_MODALITY)

In [6]:
def swaps_for(benchmark):
    """Use the configured stability replacements or the full swap list for other benchmarks."""
    return STABILITY_SWAPS if benchmark == "mrna_stability" else SWAPS


def required_encoders(benchmark):
    """List the original and replacement encoders needed for one benchmark."""
    if EXPERIMENT == "ecoli_ernie":
        return [ORIGINAL["dna"], ORIGINAL["prot"], *(key for _, key in swaps_for(benchmark))]
    # Remove duplicates without changing order: original streams are checked first.
    return list(dict.fromkeys([
        *ORIGINAL.values(),
        *(encoder_key for _, encoder_key in swaps_for(benchmark)),
    ]))


def cache_directory(benchmark, encoder_key):
    """Keep new RNA arrays separate from the original shared cache tree."""
    root = NEW_CACHE_ROOT if EXPERIMENT == "ecoli_ernie" and encoder_key in {"rnaernie", "ernierna"} else CACHE_ROOT
    return root / benchmark / encoder_key


def cache_is_read_only(encoder_key):
    """The focused extension may extract only its two new RNA streams."""
    return EXPERIMENT == "ecoli_ernie" and encoder_key not in {"rnaernie", "ernierna"}


def requested_variants(modality, encoder_key):
    if EXPERIMENT == "ecoli_ernie":
        return ("mil", "concat")
    return swap_results.swap_variants(
        modality, grid=tokens.grid_resolution(encoder_key) != "codon",
    )

In [7]:
def result_recipe(seed):
    """Match the full retained TrainConfig, including defaults not exposed above."""
    return vars(train.TrainConfig(
        batch_size=BATCH_SIZE,
        max_epochs=MAX_EPOCHS,
        channels=CHANNELS,
        tconv=TCONV,
        seed=seed,
    ))


# Plan every requested fit before extraction. Reuse actual saved paths, including
# compatible historical names; a missing result is different from a conflicting file.
def requested_identity(benchmark, variant, config, mapping, *, rows=None, verification=None):
    """Bind the fit to ordered rows and cache metadata without opening token arrays."""
    if rows is None:
        rows = datasets.blf_benchmark_rows(benchmark)
    spec = datasets.DATASETS[benchmark]
    records = {
        modality: run_identity.cache_record(
            cache_directory(benchmark, encoder_key), verification=verification,
        )
        for modality, encoder_key in mapping.items()
    }
    return run_identity.blf_identity(
        rows=rows, benchmark=benchmark, variant=variant, config=config,
        encoders=mapping, caches=records, task=spec.task,
        label_column=spec.label_column, split_column=spec.split_column,
    )


def execution_plan(benchmark):
    """Use the same preflight before extraction and immediately before fitting."""
    focused = EXPERIMENT == "ecoli_ernie"
    return swap_plan.plan_benchmark(
        benchmark=benchmark,
        config=result_recipe(SEED),
        selection_seed=SELECTION_SEED,
        original_encoders=ORIGINAL,
        original_folders=ORIGINAL_RESULTS[benchmark],
        original_destination=ORIGINAL_RESULTS[benchmark][-1],
        swaps=[
            (modality, encoder_key, tokens.grid_resolution(encoder_key) != "codon")
            for modality, encoder_key in swaps_for(benchmark)
        ],
        swap_root=RESULTS_ROOT / benchmark,
        selection_original_folders=(
            RESULT_PATHS.selection_originals[benchmark]
            if RESULT_PATHS.fixed_historical_selection else None
        ),
        selection_snapshot=RESULT_PATHS.selection_snapshot,
        selection_swap_root=(
            RESULT_PATHS.selection_swaps / benchmark
            if RESULT_PATHS.fixed_historical_selection else None
        ),
        checkpoint_validator=(
            None if READ_SAVED_RESULTS else CHECKPOINT_VALIDATOR or train.checkpoint_preflight
        ),
        identity_factory=None if READ_SAVED_RESULTS else requested_identity,
        fit_variants=("mil", "concat") if focused else None,
        include_originals=not focused,
        select_singles=not focused,
    )


In [8]:
FIT_PLANS = {benchmark: execution_plan(benchmark) for benchmark in BENCHMARKS}
fit_inventory = pd.DataFrame([row for plan in FIT_PLANS.values() for row in plan.rows()])
if not fit_inventory.empty:
    display(fit_inventory.groupby(["benchmark", "role", "status"]).size().rename("fits"))
    display(fit_inventory.loc[fit_inventory["status"] != "reuse"])
for benchmark, plan in FIT_PLANS.items():
    for issue in plan.issues:
        print(f"{benchmark}: {issue}")

if not READ_SAVED_RESULTS:
    # Stop the entire request before expensive work if any comparison is blocked.
    for plan in FIT_PLANS.values():
        plan.require_ready()

PENDING_RUNS = [
    run for plan in FIT_PLANS.values() for run in plan.runs
    if run.status in ("new", "resume")
]
PENDING_CACHE_KEYS = {
    (run.benchmark, encoder_key)
    for run in PENDING_RUNS for encoder_key in run.encoders.values()
}
print(f"Training seed {SEED}; selection seed {SELECTION_SEED}; "
      f"{len(PENDING_RUNS)} new/resumable fits in the displayed plan.")


Training seed 0; selection seed 0; 0 new/resumable fits in the displayed plan.


benchmark       role         status
ecoli_proteins  replacement  reuse     4
Name: fits, dtype: int64

,benchmark,role,encoder,variant,seed,lambda,status,reason,source,destination,legacy identity


In [9]:
def result_variant_label(result):
    """Keep unregularized MIL separate from the selected positive-penalty variant."""
    return (
        "mil (lam 0)"
        if result.variant == "mil" and result.config["entropy_lambda"] == 0
        else result.variant
    )


def original_report(benchmark, seed):
    return swap_results.read_results(
        ORIGINAL_RESULTS[benchmark],
        benchmark=benchmark,
        config=result_recipe(seed),
        encoders=ORIGINAL,
        native_fallback=True,
    )


def original_runs(benchmark, seed=None):
    """Read the selected seed; ambiguous identities cannot silently replace one another."""
    report = original_report(benchmark, SEED if seed is None else seed)
    # Duplicated identities are withheld by the reader; other saved scores still stand.
    records = report.records if READ_SAVED_RESULTS else report.require_unique()
    return [train.Result.load(record["path"]) for record in records]


def selected_mil_penalty(benchmark):
    """Reuse the complete original positive-penalty grid's seed-0 selection."""
    mil_penalty = FIT_PLANS[benchmark].lambda_
    if mil_penalty is None:
        raise RuntimeError(f"{benchmark}: seed-0 MIL selection unavailable; inspect the execution plan")
    return mil_penalty


In [10]:
# E. coli accuracy is displayed as a percentage; score differences are percentage points.
SCALE = {"ecoli_proteins": 100.0}
HEADS = swap_results.HEADS


def swap_runs(benchmark, encoder_key, mil_penalty, seed=None):
    """Read only the selected seed and inherited recipe, refusing duplicate identities."""
    modality = next(
        slot for slot, replacement_encoder in swaps_for(benchmark)
        if replacement_encoder == encoder_key
    )
    report = swap_report(
        benchmark, modality, encoder_key, SEED if seed is None else seed, mil_penalty,
    )
    records = report.records if READ_SAVED_RESULTS else report.require_unique()
    return [train.Result.load(result["path"]) for result in records]


def results_for_encoder_set(benchmark, modality=None, encoder_key=None, seed=None):
    """Collect unique variant records for the original or a replacement triple."""
    try:
        mil_penalty = selected_mil_penalty(benchmark)
    except RuntimeError:
        if not READ_SAVED_RESULTS:
            raise
        # Missing MIL selection does not hide independent single/concat/cross scores.
        mil_penalty = None
    runs = {}
    # Keep the validation-selected positive MIL penalty among the original runs.
    for result in original_runs(benchmark, seed):
        name = result_variant_label(result)
        if name == "mil" and result.config["entropy_lambda"] != mil_penalty:
            continue
        runs[name] = result
    if encoder_key is None:
        return runs

    # Retain unchanged-slot singles; replace this slot's baselines and compared heads.
    for name in [
        variant for variant in runs
        if variant in HEADS or variant.endswith(f":{modality}")
    ]:
        del runs[name]
    for result in swap_runs(benchmark, encoder_key, mil_penalty, seed):
        runs[result_variant_label(result)] = result
    return runs


In [11]:
def extension_head_scores(benchmark, directory, encoder_key, seed, penalty):
    """Read the two head accuracies for one encoder and training seed."""
    report = swap_results.read_results(
        [directory / benchmark / encoder_key], benchmark=benchmark,
        config=result_recipe(seed), encoders={**ORIGINAL, "rna": encoder_key},
        mil_lambda=penalty, variants=("mil", "concat"),
    )
    return {entry["data"]["variant"]: 100 * entry["data"]["test_score"]
            for entry in report.require_unique()}


def extension_score_table(benchmark, seeds):
    """Compare the two extension encoders with the earlier Newton RiNALMo fits."""
    penalty = selected_mil_penalty(benchmark)
    rows = []
    for encoder_key, directory, evidence in [
        ("rinalmo", REFERENCE_SWAP_ROOT, "existing RiNALMo reference"),
        *((key, RESULTS_ROOT, "focused extension") for _, key in swaps_for(benchmark)),
    ]:
        scores = {seed: extension_head_scores(benchmark, directory, encoder_key, seed, penalty)
                  for seed in seeds}
        for variant, head in (("mil", "MIL gated"), ("concat", "Concatenation")):
            row = {"RNA encoder": ENCODERS[encoder_key].label, "head": head, "evidence": evidence}
            for seed in seeds:
                row[f"Seed {seed} (%)"] = scores[seed].get(variant, np.nan)
            rows.append(row)
    table = pd.DataFrame(rows).set_index(["RNA encoder", "head", "evidence"])
    if tuple(seeds) == (0, 1, 2):
        table["Mean (%)"] = table.mean(axis=1, skipna=False)
    return table


def benchmark_score_table(benchmark):
    """Build one benchmark's score and gain rows from saved results."""
    scale = SCALE.get(benchmark, 1.0)
    original = results_for_encoder_set(benchmark)
    rows = []
    # The (None, None) entry supplies the original-triple reference row first.
    for modality, encoder_key in [(None, None), *swaps_for(benchmark)]:
        runs = results_for_encoder_set(benchmark, modality, encoder_key)
        selection_runs = (
            runs if SEED == SELECTION_SEED
            else results_for_encoder_set(
                benchmark, modality, encoder_key, seed=SELECTION_SEED,
            )
        )
        selected_single = validation_selected_single(
            runs, modality, encoder_key, selection_runs, benchmark=benchmark,
        )
        # No replacement-only score exists for the original row; unavailable scores stay NaN.
        row = {
            "slot": modality or "none",
            "encoder": (
                ENCODERS[encoder_key].label if encoder_key else "BioLangFusion's own"
            ),
            "alone": (
                runs.get(f"unimodal:{modality}").test_score * scale
                if modality and f"unimodal:{modality}" in runs else np.nan
            ),
            "best single": selected_single,
            "best single score": (
                runs[selected_single].test_score * scale if selected_single else np.nan
            ),
        }
        for head in HEADS:
            score = runs[head].test_score * scale if head in runs else np.nan
            row[head] = score
            # Distinguish fusion benefit from the overall effect of replacing an encoder.
            row[f"{head} Δfusion"] = score - row["best single score"]
            row[f"{head} vs original"] = (
                score - original[head].test_score * scale
                if head in original and encoder_key else np.nan
            )
        rows.append(row)
    return pd.DataFrame(rows).set_index(["slot", "encoder"])

## Held-out accuracy across training seeds

Each seed trains a new prediction head. The mean uses all three unrounded accuracy values. Variation across these seeds describes training variability on the fixed split.

In [12]:
for benchmark in BENCHMARKS:
    print(f"\n{benchmark}" + (" (accuracy x 100)" if benchmark in SCALE else " (Spearman)"))
    try:
        if EXPERIMENT == "ecoli_ernie":
            seeds = (0, 1, 2) if READ_SAVED_RESULTS else (SEED,)
            extension_scores = extension_score_table(benchmark, seeds)
            display(extension_scores.round(2))
        else:
            display(benchmark_score_table(benchmark).round(3))
    except (OSError, RuntimeError, ValueError, KeyError, TypeError) as missing:
        print("  not ready:", missing)


ecoli_proteins (accuracy x 100)


Seed 0 (%)  Seed 1 (%)  Seed 2 (%)  Mean (%)
RNA encoder            head          evidence                                                                
RiNALMo 150M           MIL gated     existing RiNALMo reference       59.41       59.27       58.57     59.08
                       Concatenation existing RiNALMo reference       59.83       61.38       59.13     60.11
RNAErnie (author 2048) MIL gated     focused extension                56.18       59.27       56.32     57.26
                       Concatenation focused extension                57.72       57.72       56.18     57.21
ERNIE-RNA              MIL gated     focused extension                54.92       54.92       56.46     55.43
                       Concatenation focused extension                58.01       56.18       58.43     57.54